# 02 Exploration

Training and test runs.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

import lightgbm as lgbm

from inf_minesweeper.loader import load, split_by_game
from inf_minesweeper.features import one_hot, handcrafted
from inf_minesweeper.models import Constant, LocalRatio, LogisticFeatures
from inf_minesweeper.metrics import evaluate

d = load("../data/raw/guesses.npz")
W, y, q, game, pos = d["window"], d["mine"], d["q"], d["game"], d["position"]

train, test = split_by_game(game)                       # 4,000 / 1,000 games
# hold out some training games for validation
fit, val = split_by_game(np.where(train, game, -1), test_frac=0.125, seed=1)
fit, val = fit & train, val & train

baseline = Constant().fit(W[train], y[train]).predict(W[test])
results = {}

def score(name, p):
    results[name] = evaluate(y[test], p, q[test], game[test], pos[test], baseline)
    return pd.DataFrame(results).T.round(3)

## Baselines

In [ ]:
for name, model in {
    "constant, density 0.21": Constant(0.21),
    "constant, mine rate": Constant(),
    "local ratio": LocalRatio(),
    "logistic, 8 features": LogisticFeatures(),
}.items():
    score(name, model.fit(W[train], y[train]).predict(W[test]))
pd.DataFrame(results).T.round(3)

## Gradient-boosted trees

In [ ]:


# for k in (3, 5, 7):
#     X_fit, X_val = tree_features(W[fit], k), tree_features(W[val], k)
#     model = lgbm.LGBMClassifier(
#         n_estimators=3000, 
#     )